In [19]:
%load_ext autoreload
%autoreload 2

# Importing dependencies
# import torch
# from PIL import Image
# from torch import nn,save,load
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
# import torch.nn.functional as F


from noise import *
from pathlib import Path 
import json
from data import *
from trackers import *
from main import *

from covariance_generation import *

from networks.conditioning import *

import lpips

seed = 20
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# Loading Data
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="/mnt/home/nzilberstein/datasets", download=False, train=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

def load_args(name, step="last", log=True, dataloaders=False):
    """ Load an experiment with a given name. step can be an integer, "best", or "last" (default). """
    exp_dir = Path("models") / name

    with open(exp_dir / "args.json") as f:
        args_dict = json.load(f)

    return args_dict


def load_exp(name, step="last", log=True, dataloaders=False):
    """ Load an experiment with a given name. step can be an integer, "best", or "last" (default). """
    exp_dir = Path("models") / name

    with open(exp_dir / "args.json") as f:
        args_dict = json.load(f)

    args_dict['size_network'] = "small"

    ctx = TrainingContext(**args_dict, step=step, key_remap=None, seed=None, dataloaders=dataloaders, writer=False)
    if log:
        print(f"{name}: retrieved model at step {ctx.step}")

    # Disable DataParallel (needed for Hessian computation)
    # ctx.model.network = ctx.model.network.module

    # Put in eval mode and disable gradients with respect to all parameters.
    ctx.model.eval()
    for p in ctx.model.parameters():
        p.requires_grad = False

    # Normalize energies.
    # ctx.network.network.log_normalization_constant = ctx.test_perf.log_normalization_constant

    return ctx

def get_sigmas(sigma_begin, sigma_end, num_classes, device, sigma_dis):
    if sigma_dist == 'geometric':
        sigmas = torch.tensor(
            np.exp(np.linspace(np.log(sigma_begin), np.log(sigma_end),
                               num_classes))).float().to(device)
    elif sigma_dist == 'uniform':
        sigmas = torch.tensor(
            np.linspace(sigma_begin, sigma_end, num_classes)
        ).float().to(device)

    else:
        raise NotImplementedError('sigma distribution not supported')

    return sigmas

torch.set_default_dtype(torch.float32)
    # torch.set_printoptions(precision=10, sci_mode=False)

# args = load_args("multigpu/combined_cov_tworandvars/energy_song_anisoEmb_groupNorm_lambda0_lr1.5e-4_lrdecay15000_1000warmup_d2", step = "best")
args = load_args("multigpu/inpainting_mnist/energy_songSmall_autoregressive_MNIST", step = "best")
# args = load_args("multigpu/inpainting_celeba/energy_song_autoregressive_Celeba", step = "best")
# args = load_args("multigpu/combined_cov_tworandvars/energy_songWavelet_anisoEmb_groupNorm_lambda1_lr1.5e-4_lrdecay15000_1000warmup_d2", step = "last")
step = "last"
ctxs = {
    # "energy-non-reg": load_exp("multigpu/combined_cov_tworandvars/energy_song_anisoEmb_groupNorm_lambda0_lr1.5e-4_lrdecay15000_1000warmup_d2", step = step),
    # "Energy-Dual": load_exp("multigpu/combined_cov_tworandvars/energy_song_anisoEmb_groupNorm_lambda1_mult_lr2e-4_lrdecay50000_1000warmup_d2", step = step),
    # "Energy-Dual-100k": load_exp("multigpu/combined_cov_tworandvars/energy_song_anisoEmb_groupNorm_lambda1_mult_lr2e-4_lrdecay50000_1000warmup_d2", step = 100000),
    # "Denoiser": load_exp("multigpu/inpainting_celeba/denoiser_songLarge_score_anisoEmb_groupNorm_mult_lr2e-4_lrdecay50000_1000warmup", step = step),
    "Energy-Dual": load_exp("multigpu/inpainting_mnist/energy_songSmall_autoregressive_MNIST", step = step),
    # "Energy-Dual": load_exp("multigpu/inpainting_celeba/energy_song_autoregressive_Celeba", step = "last"),
}

default_ctx = ctxs["Energy-Dual"]
device = default_ctx.device
dataset_info = default_ctx.dataset_info
d = dataset_info.dimension


# Load data
test_batch_size = 1024
H = 28
CHW = 3 * H * H

train_dataloader, test_dataloader, dataset_info = load_data(
    dataset=args["dataset"], spatial_size=args["spatial_size"], grayscale=args["grayscale"], horizontal_flip=False, data_subset=eval(args["data_subset"]),
    train_batch_size=test_batch_size, test_batch_size=test_batch_size, num_workers=args["num_workers"], seed=2
)

images = next(iter(test_dataloader))  # load for testing things.

shape = (images[0].shape[0], CHW)

time_tracker: TimeTracker = TimeTracker()
time_tracker.switch("initialization")


loss_fn_alex = lpips.LPIPS(net='alex').cuda() # best forward scores

multigpu/inpainting_mnist/energy_songSmall_autoregressive_MNIST: retrieved model at step 300000
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /mnt/home/nzilberstein/venvs/diffusion/lib64/python3.11/site-packages/lpips/weights/v0.1/alex.pth


In [27]:
dataset1 = train_dataloader.dataset
dataset2 = train_loader.dataset

dataset1, dataset2

(Dataset MNIST
     Number of datapoints: 60000
     Root location: /mnt/home/nzilberstein/datasets
     Split: Train
     StandardTransform
 Transform: Compose(
                  ToImage()
                  ToDtype(scale=True)
            ),
 Dataset MNIST
     Number of datapoints: 60000
     Root location: /mnt/home/nzilberstein/datasets
     Split: Train
     StandardTransform
 Transform: Compose(    ToTensor()))

In [3]:
# Define the image classifier model
class ImageClassifier(nn.Module):
    def __init__(self):
        super(ImageClassifier, self).__init__()
        
        # 1st block: 1 input channel (for grayscale MNIST) -> 32 output channels
        # Kernel size 3x3, Padding 1 ensures output size remains 28x28 before pooling
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        
        # 2nd block: 32 -> 64 channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        # Dropout layer for regularization
        self.dropout1 = nn.Dropout(0.25)
        
        # 3rd block: 64 -> 128 channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Final dropout before fully connected layers
        self.dropout2 = nn.Dropout(0.5)
        
        # After two MaxPool2d (kernel_size=2) operations, the spatial size 
        # is reduced from 28x28 -> 14x14 -> 7x7. 
        # The output of the last conv layer is 128 channels, so 128 * 7 * 7
        self.fc1 = nn.Linear(128 * 7 * 7, 512) 
        self.fc2 = nn.Linear(512, 10) # 10 output classes (digits 0-9)

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2) # Size becomes 14x14
        
        # Block 2
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2) # Size becomes 7x7
        x = self.dropout1(x)
        
        # Block 3
        x = F.relu(self.conv3(x))
        
        # Flatten for the fully connected layers
        x = torch.flatten(x, 1)
        
        # Fully Connected Layers
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        
        # The loss function (nn.CrossEntropyLoss) typically includes Softmax, 
        # so we return the raw logits.
        return x

In [4]:
# Create an instance of the image classifier model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = ImageClassifier().to('cuda')

In [9]:
# Define the optimizer and loss function
optimizer = Adam(classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [28]:
# Train the model
for epoch in range(30):  # Train for 10 epochs
    print(epoch)
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # Reset gradients
        outputs = classifier(images)  # Forward pass
        loss = loss_fn(outputs, labels)  # Compute loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

    print(f"Epoch:{epoch} loss is {loss.item()}")

0


KeyboardInterrupt: 

In [28]:
# Save the trained model
torch.save(classifier.state_dict(), 'model_state.pt')

In [29]:
# Load the saved model
with open('model_state.pt', 'rb') as f: 
     classifier.load_state_dict(load(f))  
       

In [32]:
# Perform inference on an image
# test_dataset = datasets.MNIST(root="/mnt/home/nzilberstein/datasets", download=False, train=False, transform=transform)
# test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)
img = next(iter(test_dataloader))
img_tensor = img[0].to(device)
output = classifier(img_tensor)
print(output.shape)
predicted_label = torch.argmax(output, dim = 1)
print(f"Predicted label: {(predicted_label != img[1].to(device)).sum() / 1024}")

torch.Size([1024, 10])
Predicted label: 0.01171875
